# 05 — QAT Test-Set Inference (seed 42)

Evaluate QAT (INT8 via ModelOpt) checkpoints across all input bit-depths on the
**test set** (`/home/pf4636/imagenet2`).

QAT checkpoints: `training/checkpoints/qat/int8_in{1,2,4,8}b/`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [2]:
import json
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import modelopt.torch.opt as mto

from dataclasses import replace
from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_imagenet_dataset
from src.eval import evaluate
from quantize import get_model, get_quant_cfg, quantize_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [ ]:
QAT_CKPT_ROOT = PROJECT_ROOT / "checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
TEST_ROOT     = "/home/pf4636/imagenet2"
INPUT_BITS    = [8, 4, 2, 1]
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoints = {}
for bits in INPUT_BITS:
    run_name = f"int8_in{bits}b"
    ckpt_dir = QAT_CKPT_ROOT / run_name / "seed_42"
    fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / "seed_42" / "best.pth"
    checkpoints[bits] = {
        "weights": ckpt_dir / "qat_modelopt_best.pth",
        "mostate": ckpt_dir / "qat_modelopt_best_mostate.pt",
        "fp32_ckpt": fp32_ckpt,
    }
    assert checkpoints[bits]["weights"].exists(), f"Missing: {checkpoints[bits]['weights']}"
    assert checkpoints[bits]["mostate"].exists(), f"Missing: {checkpoints[bits]['mostate']}"
    assert fp32_ckpt.exists(), f"Missing: {fp32_ckpt}"

print("QAT checkpoints:")
for bits, paths in checkpoints.items():
    print(f"  in{bits}b: {paths['weights'].name}  (FP32 base: {paths['fp32_ckpt']})")
print(f"Test set: {TEST_ROOT}")

In [4]:
def load_qat_model(bits: int) -> nn.Module:
    """Build base model from matching FP32 checkpoint, restore modelopt quant graph, load QAT weights."""
    fp32_ckpt = str(checkpoints[bits]["fp32_ckpt"])
    model = get_model(fp32_ckpt, num_classes=100)
    restore_modelopt_state(model, str(checkpoints[bits]["mostate"]))
    ckpt = torch.load(checkpoints[bits]["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    return model

def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_ROOT)
    dataset = build_imagenet_dataset(test_cfg, split="val")
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## QAT INT8 — Test-Set Evaluation

In [5]:
SKIP_EXISTING = True
OUT_DIR = Path("../runs/qat_test").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

records = []
criterion = nn.CrossEntropyLoss()

for bits in INPUT_BITS:
    run_id = f"resnet18_qat_int8_in{bits}b_cuda_bs1"
    result_path = OUT_DIR / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  in{bits}b: exists, skipping")
        with open(result_path) as f:
            records.append(json.load(f))
        continue

    print(f"\n--- QAT INT8, input_bits={bits} ---")
    cfg = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        input_quant_bits=bits,
        model_precision="fp32",
    )
    set_seed(cfg)

    model = load_qat_model(bits)
    test_loader = build_test_loader(cfg)

    t0 = time.perf_counter()
    tracker = evaluate(model, test_loader, cfg, criterion=criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {"qat_precision": "int8", "input_quant_bits": bits, "seed": 42},
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    save_path = OUT_DIR / run_id / "result.json"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    records.append(payload)
    del model
    torch.cuda.empty_cache()

print(f"\n{len(records)} runs complete.")

  in8b: exists, skipping
  in4b: exists, skipping
  in2b: exists, skipping
  in1b: exists, skipping

4 runs complete.


## Results Summary

In [6]:
rows = []
for rec in records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    rows.append({
        "precision": "qat_int8",
        "input_bits": bits,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "lat_std": r.get("infer_ms_std", None),
        "throughput": r.get("throughput_infer_sps", None),
    })

df = pd.DataFrame(rows).sort_values("input_bits", ascending=False).reset_index(drop=True)
df

,precision,input_bits,top1,top5,lat_ms,lat_std,throughput
0,qat_int8,8,80.426,97.234,8.942,2.121,111.837
1,qat_int8,4,80.213,96.809,8.500,2.002,117.643
2,qat_int8,2,75.319,93.191,9.770,2.179,102.354
3,qat_int8,1,61.915,89.149,8.929,2.026,111.989


## Compare QAT vs PTQ Baseline

In [ ]:
ptq_results = pd.read_json(PROJECT_ROOT / "resultsv2" / "test_final_results.json")

fp32_baseline = ptq_results[
    (ptq_results["backend"] == "pytorch") &
    (ptq_results["precision"] == "fp32")
][["input_bits", "top1_mean", "top5_mean"]].rename(
    columns={"top1_mean": "fp32_top1", "top5_mean": "fp32_top5"}
)

comparison = df.merge(fp32_baseline, on="input_bits", how="left")
comparison["top1_delta"] = comparison["top1"] - comparison["fp32_top1"]
comparison["top5_delta"] = comparison["top5"] - comparison["fp32_top5"]
comparison

## Save Results

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_test_results_seed42.json"
df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")